# 7. scikit-learn 지도학습

> **제7장** · **이론편 대응: 8장 (Machine Learning 기초), 9.3절 (모델 평가)**
> **예상 소요**: 50분
> **필요 사양**: CPU만으로 충분

---

## 이 장에서 하는 일

04장에서 직접 만든 선형회귀를 **라이브러리로 다시 해 보고**, 분류 문제로 넘어간다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 04번 코드 vs scikit-learn | 8.2절 |
| 2 | 데이터 분할 — 훈련·검증·시험 | 11.6절 |
| 3 | 분류 문제와 결정 경계 | 8.3절 |
| 4 | **혼동행렬과 지표 계산** | 9.3절 |
| 5 | **정확도의 함정** | 9.3절 |
| 6 | 결정 트리 | 9.1절 |

**5절이 중요하다.** 이론편 9.3절에서 "아무것도 안 하는 분류기가 정확도 91%"라고 했던 것을
직접 만들어 확인한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import sklearn

# 한글 폰트 (03장 참조)
_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

print(f"scikit-learn 버전: {sklearn.__version__}")
print("준비 완료")

---

## 1. 04번 코드와 scikit-learn 비교

04장에서 우리는 경사하강법으로 선형회귀를 구현했다. 같은 데이터를 scikit-learn으로 풀어 보고
결과가 같은지 확인한다.

**두 가지를 확인한다.**
1. 결과가 04장·정규방정식과 일치하는가
2. 코드가 얼마나 짧아지는가

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# 04장과 완전히 같은 데이터
rng = np.random.RandomState(42)
n = 100
X = rng.uniform(20, 120, n)
TRUE_W, TRUE_B = 0.45, 10.0
y = TRUE_W * X + TRUE_B + rng.randn(n) * 6

# --- scikit-learn 방식 ---
X_2d = X.reshape(-1, 1)          # sklearn은 2차원 입력을 요구한다
model = LinearRegression()
model.fit(X_2d, y)

w_sk = model.coef_[0]
b_sk = model.intercept_

# --- 04장에서 구한 정규방정식 해 ---
X_design = np.stack([X, np.ones(n)], axis=1)
w_opt, b_opt = np.linalg.lstsq(X_design, y, rcond=None)[0]

print("=" * 60)
print("scikit-learn vs 04번 결과")
print("=" * 60)
print(f"{'':20}{'w':<16}{'b'}")
print("-" * 60)
print(f"{'scikit-learn':20}{w_sk:<16.6f}{b_sk:.6f}")
print(f"{'정규방정식(04장)':20}{w_opt:<16.6f}{b_opt:.6f}")
print(f"{'참값':20}{TRUE_W:<16.6f}{TRUE_B:.6f}")
print("-" * 60)

assert np.allclose([w_sk, b_sk], [w_opt, b_opt]), "결과가 다릅니다"
print("[OK] 04번 결과와 일치")
print()
print("scikit-learn의 LinearRegression은 내부적으로 정규방정식(최소제곱)을 쓴다.")
print("경사하강법이 아니므로 학습률·에폭 설정이 필요 없고, 표준화도 필수가 아니다.")

### 코드 길이 비교

| 방식 | 필요한 것 | 줄 수 |
|---|---|---|
| 04번 직접 구현 | 손실함수·그래디언트·학습 루프·표준화·환산 | 약 40줄 |
| scikit-learn | `LinearRegression().fit(X, y)` | 2줄 |

**그렇다면 04장은 시간 낭비였을까?** 아니다. 두 가지 이유가 있다.

첫째, **왜 표준화가 필요한지** 알게 되었다. sklearn의 `LinearRegression`은 정규방정식을 쓰므로
표준화가 필수가 아니지만, 경사하강법을 쓰는 모델(신경망 포함)에서는 여전히 필요하다.

둘째, **라이브러리가 없는 상황을 다룰 수 있다.** 12장에서 역전파를 직접 구현할 때,
04장에서 익힌 방식이 그대로 쓰인다.

---

## 2. 데이터 분할 — 이론편 11.6절

지금까지는 데이터 전체로 학습하고 같은 데이터로 평가했다. 이론편 11.6절에서 다뤘듯
**이러면 성능을 과대평가하게 된다.**

scikit-learn의 `train_test_split`으로 나눈다.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 훈련 70% / 임시 30%
X_train, X_temp, y_train, y_temp = train_test_split(
    X_2d, y, test_size=0.3, random_state=42)

# 임시 30%를 검증 15% / 시험 15%로 다시 나눈다
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42)

print("=" * 55)
print("데이터 분할 (이론편 11.6절)")
print("=" * 55)
total = len(X_2d)
for name, arr in [("훈련", X_train), ("검증", X_val), ("시험", X_test)]:
    print(f"  {name} : {len(arr):3}개  ({len(arr)/total*100:.0f}%)")
print(f"  합계 : {total}개")
print()

# 훈련 데이터로만 학습
model = LinearRegression().fit(X_train, y_train)

print("각 데이터에서의 손실 (MSE)")
print("-" * 55)
for name, Xs, ys in [("훈련", X_train, y_train),
                     ("검증", X_val, y_val),
                     ("시험", X_test, y_test)]:
    mse = mean_squared_error(ys, model.predict(Xs))
    print(f"  {name} : {mse:8.3f}")
print("-" * 55)
print()
print("세 값이 비슷하면 정상이다. 훈련만 낮고 나머지가 높으면 과대적합이다.")

### `random_state`를 지정하는 이유

`train_test_split`은 데이터를 무작위로 나눈다. `random_state`를 지정하지 않으면
**실행할 때마다 다른 결과**가 나온다.

이론편 11.6절에서 다룬 재현성 문제가 여기서도 나타난다. 실험을 비교하려면 분할이 같아야 하므로
값을 고정해 두는 것이 좋다.

> 다만 **시험 데이터는 정말 마지막에 한 번만** 쓴다. 위에서 시험 손실을 출력한 것은 설명을 위해서이고,
> 실제로는 모델을 고르고 조정하는 동안 검증 데이터만 봐야 한다.

---

## 3. 분류 문제 — 이론편 8.3절

지금까지는 숫자를 예측하는 회귀였다. 이번에는 **범주를 맞히는** 분류를 다룬다.

이론편 8.3절에서 다룬 로지스틱 회귀를 쓴다. 이름에 "회귀"가 들어가지만 분류 모델이다.
선형 결합의 결과를 시그모이드(이론편 10.3절)에 통과시켜 0~1 사이 확률로 만들기 때문이다.

$$P(y=1 \mid x) = \sigma(w^\top x + b)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 두 덩어리로 나뉜 데이터 만들기
rng = np.random.RandomState(0)
n_per = 100

# 클래스 0: 왼쪽 아래 / 클래스 1: 오른쪽 위
c0 = rng.randn(n_per, 2) * 1.0 + np.array([-1.5, -1.0])
c1 = rng.randn(n_per, 2) * 1.0 + np.array([ 1.5,  1.5])

Xc = np.vstack([c0, c1])
yc = np.hstack([np.zeros(n_per), np.ones(n_per)]).astype(int)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.3, random_state=42, stratify=yc)

# 표준화 — 04장에서 배운 대로
scaler = StandardScaler().fit(Xc_train)
Xc_train_s = scaler.transform(Xc_train)
Xc_test_s = scaler.transform(Xc_test)

clf = LogisticRegression().fit(Xc_train_s, yc_train)

print("=" * 55)
print("로지스틱 회귀")
print("=" * 55)
print(f"학습 데이터: {len(Xc_train)}개")
print(f"계수 w     : {clf.coef_[0].round(4)}")
print(f"절편 b     : {clf.intercept_[0]:.4f}")
print(f"훈련 정확도 : {clf.score(Xc_train_s, yc_train):.4f}")
print(f"시험 정확도 : {clf.score(Xc_test_s, yc_test):.4f}")
print()

# 확률 예측 — 분류 모델은 확률도 준다
probs = clf.predict_proba(Xc_test_s)[:5]
print("앞 5개 시험 데이터의 예측 확률")
print(f"{'':6}{'클래스0':<12}{'클래스1':<12}{'예측':<8}{'실제'}")
for i, p in enumerate(probs):
    pred = int(p[1] > 0.5)
    print(f"  {i:<4}{p[0]:<12.4f}{p[1]:<12.4f}{pred:<8}{yc_test[i]}")

### `stratify`가 하는 일

`train_test_split(..., stratify=yc)`를 썼다. 이것은 **클래스 비율을 유지하며** 나누라는 뜻이다.

지정하지 않으면 우연히 한쪽 클래스가 훈련에 몰릴 수 있다. 특히 클래스가 불균형할 때
(예: 스팸 9% / 정상 91%) 이 옵션이 없으면 시험 데이터에 스팸이 거의 없는 상황도 생긴다.

분류 문제에서는 거의 항상 지정하는 것이 좋다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# --- 왼쪽: 데이터와 결정 경계 ---
ax = axes[0]
h = 0.02
x_min, x_max = Xc[:, 0].min() - 1, Xc[:, 0].max() + 1
y_min, y_max = Xc[:, 1].min() - 1, Xc[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
grid = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
Z = clf.predict(grid).reshape(xx.shape)

ax.contourf(xx, yy, Z, alpha=0.15, cmap="coolwarm")
ax.scatter(c0[:, 0], c0[:, 1], s=25, alpha=0.7, label="클래스 0", color="#1E40AF")
ax.scatter(c1[:, 0], c1[:, 1], s=25, alpha=0.7, label="클래스 1", color="#EA580C")
ax.set_title("결정 경계")
ax.set_xlabel("특성 1")
ax.set_ylabel("특성 2")
ax.legend()
ax.grid(alpha=0.3)

# --- 오른쪽: 확률 등고선 ---
ax = axes[1]
Z_prob = clf.predict_proba(grid)[:, 1].reshape(xx.shape)
cs = ax.contourf(xx, yy, Z_prob, levels=20, cmap="coolwarm", alpha=0.7)
ax.contour(xx, yy, Z_prob, levels=[0.5], colors="black", linewidths=2)
plt.colorbar(cs, ax=ax, label="클래스 1일 확률")
ax.scatter(Xc[:, 0], Xc[:, 1], s=12, c="black", alpha=0.4)
ax.set_title("예측 확률 (검은 선 = 0.5)")
ax.set_xlabel("특성 1")

plt.tight_layout()
plt.show()

print("오른쪽 그림에서 색이 진할수록 확률이 1에 가깝다.")
print("검은 선(확률 0.5)이 왼쪽 그림의 결정 경계와 같은 위치다.")

---

## 4. 혼동행렬과 지표 — 이론편 9.3절 값 검증 ★

이론편 9.3절에서 스팸 분류 예제로 지표를 손계산했다. 그 값을 그대로 재현한다.

| | 실제 스팸 | 실제 정상 |
|---|---|---|
| **스팸으로 예측** | TP = 80 | FP = 20 |
| **정상으로 예측** | FN = 10 | TN = 890 |

이론편에서 구한 값은 다음과 같았다.
- 정확도 0.970 / 정밀도 0.800 / 재현율 0.889 / F1 0.842

In [ ]:
import numpy as np
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score,
                             classification_report)

# 이론편 9.3절과 같은 상황을 데이터로 만든다
TP, FP, FN, TN = 80, 20, 10, 890
y_true = np.array([1]*TP + [0]*FP + [1]*FN + [0]*TN)   # 1=스팸
y_pred = np.array([1]*TP + [1]*FP + [0]*FN + [0]*TN)

print("=" * 60)
print("이론편 9.3절 값 검증")
print("=" * 60)

cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
print("혼동행렬")
print(f"{'':16}{'예측:스팸':<12}{'예측:정상'}")
print(f"{'실제:스팸':16}{cm[0,0]:<12}{cm[0,1]}")
print(f"{'실제:정상':16}{cm[1,0]:<12}{cm[1,1]}")
print()

metrics = {
    "정확도": (accuracy_score(y_true, y_pred), 0.970),
    "정밀도": (precision_score(y_true, y_pred), 0.800),
    "재현율": (recall_score(y_true, y_pred), 0.889),
    "F1":     (f1_score(y_true, y_pred), 0.842),
}

print(f"{'지표':<10}{'계산값':<14}{'이론편 값':<14}{'일치'}")
print("-" * 60)
for name, (calc, book) in metrics.items():
    ok = abs(calc - book) < 0.001
    print(f"{name:<10}{calc:<14.4f}{book:<14.3f}{'O' if ok else 'X'}")
    assert ok, f"{name}이 이론편 값과 다릅니다"
print("-" * 60)
print("[OK] 이론편 9.3절 손계산과 완전히 일치")

### 각 지표가 답하는 질문

| 지표 | 계산 | 답하는 질문 |
|---|---|---|
| 정확도 | (TP+TN) / 전체 | 전체 중 몇 개나 맞혔나 |
| 정밀도 | TP / (TP+FP) | **스팸이라 한 것 중** 진짜 스팸 비율 |
| 재현율 | TP / (TP+FN) | **진짜 스팸 중** 몇 개나 잡았나 |
| F1 | 정밀도·재현율의 조화평균 | 둘의 균형 |

정밀도 0.8은 "스팸이라 걸러낸 100통 중 20통이 멀쩡한 메일"이라는 뜻이다.
정확도 97%만 보면 훌륭해 보이지만, 실제로는 중요한 메일이 스팸함에 들어가고 있다.

---

## 5. 정확도의 함정 — 이론편 9.3절 ★

이론편 9.3절에서 **"아무것도 하지 않는 분류기가 정확도 91%"**라고 했다. 직접 만들어 확인한다.

이 분류기는 판단을 전혀 하지 않고 **모든 메일을 정상이라고 답한다.**

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score

# 아무 판단도 하지 않는 분류기: 전부 정상(0)으로 예측
y_lazy = np.zeros_like(y_true)

print("=" * 60)
print("아무것도 하지 않는 분류기")
print("=" * 60)

acc_lazy = accuracy_score(y_true, y_lazy)
rec_lazy = recall_score(y_true, y_lazy, zero_division=0)
prec_lazy = precision_score(y_true, y_lazy, zero_division=0)

print(f"{'지표':<12}{'제대로 만든 분류기':<20}{'아무것도 안 하는 분류기'}")
print("-" * 60)
print(f"{'정확도':<12}{accuracy_score(y_true, y_pred):<20.3f}{acc_lazy:.3f}  ← 높아 보인다")
print(f"{'정밀도':<12}{precision_score(y_true, y_pred):<20.3f}{prec_lazy:.3f}")
print(f"{'재현율':<12}{recall_score(y_true, y_pred):<20.3f}{rec_lazy:.3f}  ← 하나도 못 잡음")
print("-" * 60)

assert abs(acc_lazy - 0.910) < 0.001, "이론편 값(0.910)과 다릅니다"
assert rec_lazy == 0.0
print("[OK] 이론편 9.3절의 0.910과 일치")
print()
print("왜 이런 일이 생기나")
print(f"  전체 {len(y_true)}통 중 정상이 {(y_true==0).sum()}통 ({(y_true==0).mean()*100:.0f}%)")
print("  → 무조건 다수 쪽으로 찍기만 해도 그 비율만큼 맞는다")
print()
print("이는 이론편 6.2절 베이즈 예제와 같은 구조다.")
print("드문 것을 찾는 문제에서는 전체 비율이 지표를 왜곡한다.")

### 그래서 무엇을 봐야 하나

| 상황 | 중요한 지표 | 이유 |
|---|---|---|
| 암 검진 | **재현율** | 환자를 놓치는 대가가 크다 |
| 스팸 필터 | **정밀도** | 중요 메일이 걸러지면 곤란하다 |
| 불량품 검사 | **재현율** | 불량품 출하를 막아야 한다 |
| 클래스 불균형 | **F1 또는 재현율** | 정확도는 의미가 없다 |

정확도 하나만 보고 모델을 판단하면 안 된다는 것이 이론편 9.3절의 요지였고,
방금 그것을 숫자로 확인했다.

---

## 6. 결정 트리 — 이론편 9.1절

로지스틱 회귀는 **직선(또는 평면)으로만** 경계를 그을 수 있다. 이론편 9.1절에서 다룬 결정 트리는
"이 값이 3보다 큰가?" 같은 질문을 반복해 영역을 나누므로, 곡선 형태의 경계도 만들 수 있다.

두 모델을 같은 데이터에 적용해 차이를 본다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 직선으로 나눌 수 없는 데이터 (두 개의 반달 모양)
from sklearn.datasets import make_moons
Xm, ym = make_moons(n_samples=300, noise=0.25, random_state=42)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
    Xm, ym, test_size=0.3, random_state=42, stratify=ym)

sc = StandardScaler().fit(Xm_tr)
Xm_tr_s, Xm_te_s = sc.transform(Xm_tr), sc.transform(Xm_te)

models = {
    "로지스틱 회귀": LogisticRegression().fit(Xm_tr_s, ym_tr),
    # ── DecisionTreeClassifier 파라미터 ──────────────────────────
    #   criterion          분할 기준.  기본값 'gini'
    #                      'gini'(빠름) / 'entropy'(정보이득) / 'log_loss'
    #   max_depth          최대 깊이.  기본값 None(제한 없음 → 과대적합)
    #                      예: 3(단순) / 5~10(보통) / None(완전 성장)
    #   min_samples_split  분할에 필요한 최소 표본.  기본값 2
    #                      예: 10, 20 — 클수록 단순한 트리
    #   min_samples_leaf   잎에 있어야 할 최소 표본.  기본값 1
    #                      예: 5, 10 — 극단적으로 작은 잎을 막는다
    #   max_features       분할 시 고려할 특성 수.  기본값 None(전부)
    #                      'sqrt' / 'log2' / 0.5(비율) — 랜덤성 부여
    #   ccp_alpha          가지치기 강도.  기본값 0.0
    #                      예: 0.01, 0.02 — 학습 후 잘라낸다
    #   random_state       난수 시드.  기본값 None → 재현성 위해 고정
    # ──────────────────────────────────────────────────────────────
    "결정 트리 (깊이 3)": DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xm_tr, ym_tr),
    "결정 트리 (제한 없음)": DecisionTreeClassifier(random_state=42).fit(Xm_tr, ym_tr),
}

print("=" * 60)
print("모델별 성능 비교")
print("=" * 60)
print(f"{'모델':<24}{'훈련 정확도':<16}{'시험 정확도'}")
print("-" * 60)
for name, m in models.items():
    if "로지스틱" in name:
        tr = m.score(Xm_tr_s, ym_tr); te = m.score(Xm_te_s, ym_te)
    else:
        tr = m.score(Xm_tr, ym_tr); te = m.score(Xm_te, ym_te)
    gap = tr - te
    note = "  ← 격차가 크다(과대적합)" if gap > 0.1 else ""
    print(f"{name:<24}{tr:<16.4f}{te:.4f}{note}")
print("-" * 60)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

h = 0.02
x_min, x_max = Xm[:, 0].min() - 0.5, Xm[:, 0].max() + 0.5
y_min, y_max = Xm[:, 1].min() - 0.5, Xm[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid_raw = np.c_[xx.ravel(), yy.ravel()]
grid_scaled = sc.transform(grid_raw)

for ax, (name, m) in zip(axes, models.items()):
    g = grid_scaled if "로지스틱" in name else grid_raw
    Z = m.predict(g).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap="coolwarm")
    ax.scatter(Xm[ym == 0][:, 0], Xm[ym == 0][:, 1], s=15, color="#1E40AF", alpha=0.7)
    ax.scatter(Xm[ym == 1][:, 0], Xm[ym == 1][:, 1], s=15, color="#EA580C", alpha=0.7)
    ax.set_title(name)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

print("왼쪽  : 직선 하나로만 나눌 수 있어 반달 모양을 못 따라간다")
print("가운데: 깊이를 3으로 제한 — 계단 모양이지만 대체로 잘 나눔")
print("오른쪽: 제한 없음 — 훈련 데이터에 너무 맞춰졌다 (이론편 8.5절 과대적합)")

### 결정 트리의 규칙을 직접 보기

결정 트리의 장점은 **판단 과정을 사람이 읽을 수 있다**는 것이다.
이론편 9.1절에서 "해석 가능성"이라 했던 것이 이것이다.

In [ ]:
from sklearn.tree import export_text
from sklearn.tree import DecisionTreeClassifier

tree_small = DecisionTreeClassifier(max_depth=2, random_state=42).fit(Xm_tr, ym_tr)

print("=" * 55)
print("결정 트리가 학습한 규칙")
print("=" * 55)
print(export_text(tree_small, feature_names=["특성1", "특성2"]))
print()
print("이 규칙을 사람이 그대로 읽고 검토할 수 있다.")
print("신경망(10장 이후)은 이런 설명이 불가능하다 —")
print("이론편 9.1절에서 다룬 성능과 해석 가능성의 맞바꿈이다.")

---

## 7. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| 8.2 | 선형회귀 해 | 04장·정규방정식과 일치 ✓ |
| 9.3 | 정확도 0.970 / 정밀도 0.800 | 일치 ✓ |
| 9.3 | 재현율 0.889 / F1 0.842 | 일치 ✓ |
| 9.3 | **무의미 분류기 정확도 0.910** | 일치 ✓ |

### 기억할 것

| 항목 | 요점 |
|---|---|
| sklearn 입력 | 2차원이어야 함 — `X.reshape(-1, 1)` |
| `fit` / `predict` | 모든 모델이 같은 인터페이스 |
| `stratify` | 분류에서는 거의 항상 지정 |
| `random_state` | 재현성을 위해 고정 (이론편 11.6절) |
| 표준화 | **훈련 데이터로만 `fit`**, 검증·시험에는 `transform`만 |
| 정확도 | 클래스가 불균형하면 의미 없음 |

### `fit`을 훈련 데이터로만 하는 이유

`StandardScaler().fit(X_train)`처럼 훈련 데이터로만 평균·표준편차를 구했다.
시험 데이터까지 포함해 계산하면, **시험 데이터의 정보가 학습에 새어 든다.**

이를 데이터 누수(data leakage)라 하며, 성능이 실제보다 좋게 나오는 대표적 원인이다.

### 다음 장

**8. 비지도학습과 모델 평가** — 정답 없이 데이터의 구조를 찾는 방법을 다룬다.
이론편 4.4절에서 확인한 PCA를 sklearn으로 다시 하고, K-Means를 직접 구현한다.